In [1]:
import requests
from bs4 import BeautifulSoup
import time
import csv

BASE_URL = "https://carfromjapan.com/cheap-used-cars-for-sale"
PARAMS = {
    "minYear": 2018,
    "make": "toyota",
    "model": "vitz"
}
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

In [2]:
# Collection of car URLs
def get_car_links(page_url, params):
    response = requests.get(page_url, params=params, headers=HEADERS)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    car_links = []
    for h3_tag in soup.find_all("h3"):
        link = h3_tag.find("a")
        if link and link.get("href"):
            href = link["href"]
            full_url = f"https://carfromjapan.com{href}" if href.startswith("/") else href
            car_links.append(full_url)
    
    return car_links


In [12]:
# Scraping car details
def get_car_details(car_url):
    car_response = requests.get(car_url, headers=HEADERS)
    car_soup = BeautifulSoup(car_response.content, 'html.parser')
    
    car_data = {"url": car_url}
    
    specs_table = car_soup.find("table", class_="w-full table-fixed")

    price_tag = car_soup.find("span", class_="car-price")
    if price_tag:
        car_data["Price"] = price_tag.text.strip()
    else:
        car_data["Price"] = "N/A"
        specs_table = car_soup.find("table", class_="w-full table-fixed")
    if specs_table:
        for row in specs_table.find_all("tr"):
            cells = row.find_all("td")
            for i in range(0, len(cells), 2):
                if i + 1 < len(cells):
                    label = cells[i].text.strip()
                    value = cells[i + 1].text.strip()
                    car_data[label] = value
    else:
        print(f"  Specs table not found: {car_url}")
    
    return car_data

In [13]:
all_cars = []

for index, url in enumerate(all_links, start=1):
    print(f"[{index}/{len(all_links)}] Scraping: {url}")
    
    try:
        details = get_car_details(url)
        all_cars.append(details)
    except requests.exceptions.Timeout:
        print(f" Timed out, skipping: {url}")
        all_cars.append({"url": url, "error": "timeout"})
    except Exception as e:
        print(f" Error on {url}: {e}")
        all_cars.append({"url": url, "error": str(e)})
    
    time.sleep(2)

[1/250] Scraping: https://carfromjapan.com/cheap-used-daihatsu-hijet-truck-2021-for-sale-68a7c82cc49cc5ebcf1a4eae
[2/250] Scraping: https://carfromjapan.com/cheap-used-mercedes-benz-amg-2021-for-sale-6984348ec15b58c118ba33df
[3/250] Scraping: https://carfromjapan.com/cheap-used-toyota-rav4-2019-for-sale-6a0505694aa46852b18e8141
[4/250] Scraping: https://carfromjapan.com/cheap-used-toyota-rav4-2019-for-sale-69d905a20d2462329b4839c4
[5/250] Scraping: https://carfromjapan.com/cheap-used-toyota-alphard-2025-for-sale-6a06b195c15b58c118022102
[6/250] Scraping: https://carfromjapan.com/cheap-used-toyota-rav4-2019-for-sale-699327ce3a4a6f6b0377e368
[7/250] Scraping: https://carfromjapan.com/cheap-used-toyota-hiace-van-2019-for-sale-69e48aa8c15b58c118725c97
[8/250] Scraping: https://carfromjapan.com/cheap-used-daihatsu-hijet-truck-2024-for-sale-692a31d907500f365b23d1b0
[9/250] Scraping: https://carfromjapan.com/cheap-used-toyota-alphard-2025-for-sale-69f58db2c15b58c118bda9d1
[10/250] Scraping: h

In [15]:
if all_cars:
    keys = set()
    for car in all_cars:
        keys.update(car.keys())
    
    with open("car_listings.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(keys))
        writer.writeheader()
        writer.writerows(all_cars)
    print(f" Data saved to car_listings.csv — {len(all_cars)} cars total")
else:
    print(" No data collected — CSV not created")



 Data saved to car_listings.csv — 250 cars total


In [16]:
import pandas as pd

In [17]:
df = pd.read_csv("car_listings.csv")

In [18]:
df.shape
df.head()
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 19 columns):
 #   Column                                          Non-Null Count  Dtype
---  ------                                          --------------  -----
 0   url                                             250 non-null    str  
 1   Engine Capacity                                 250 non-null    str  
 2   Registration Year                               250 non-null    str  
 3   Mileage                                         250 non-null    str  
 4   VIN / Chassis No.                               250 non-null    str  
 5   Transmission                                    250 non-null    str  
 6   Model Grade                                     250 non-null    str  
 7   Drive Type                                      250 non-null    str  
 8   Dimension                                       250 non-null    str  
 9   Fuel Type                                       250 non-null    str  
 10  E

url                                               0
Engine Capacity                                   0
Registration Year                                 0
Mileage                                           0
VIN / Chassis No.                                 0
Transmission                                      0
Model Grade                                       0
Drive Type                                        0
Dimension                                         0
Fuel Type                                         0
Exterior Color                                    0
*Full VIN/Chassis no. will be shown on Invoice    0
Reference No.                                     0
Price                                             0
No. of Doors                                      0
Manufacture Year                                  0
No. of Seats                                      0
Steering                                          0
Model Code                                        0
dtype: int64

In [23]:
print("Price:", df["Price"].head(3).tolist())
print("Mileage:", df["Mileage"].head(3).tolist())
print("Engine Capacity:", df["Engine Capacity"].head(3).tolist())
print("Manufacture Year:", df["Manufacture Year"].head(3).tolist())

Price: ['US$ 6,854', 'US$ 123,747', 'US$ 18,148']
Mileage: ['56,000 km(Approx. 34,804 miles)', '10,000 km(Approx. 6,215 miles)', '14,700 km(Approx. 9,136 miles)']
Engine Capacity: ['660 cc(0.66 liters)', '4000 cc(4.00 liters)', '2000 cc(2.00 liters)']
Manufacture Year: ['-', '-', '-']


In [24]:
from datetime import datetime

In [25]:
df["Price"] = df["Price"].str.replace("US$", "", regex=False)
df["Price"] = df["Price"].str.replace(",", "", regex=False).str.strip()
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

In [26]:
df["Mileage"] = df["Mileage"].str.extract(r'([\d,]+)\s*km')
df["Mileage"] = df["Mileage"].str.replace(",", "", regex=False)
df["Mileage"] = pd.to_numeric(df["Mileage"], errors="coerce")

In [27]:
df["Engine Capacity"] = df["Engine Capacity"].str.extract(r'(\d+)\s*cc')
df["Engine Capacity"] = pd.to_numeric(df["Engine Capacity"], errors="coerce")

In [28]:
df["Registration Year"] = df["Registration Year"].str.extract(r'(\d{4})')
df["Registration Year"] = pd.to_numeric(df["Registration Year"], errors="coerce")

In [30]:
# 6. Rename columns 
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace(" ", "_", regex=False)
df.columns = df.columns.str.lower()

In [31]:
#  Removing duplicates
df = df.drop_duplicates()

In [32]:
df.to_csv("car_listings_cleaned.csv", index=False)
print("Cleaned data saved to car_listings_cleaned.csv")

Cleaned data saved to car_listings_cleaned.csv
